# Evaluación de la retroalimentación de campo

La lista de gestión dice *quién* parece estar dejando de consumir. Solo la visita dice *por qué*.
Este notebook cierra ese ciclo: cruza lo que encontraron las cuadrillas con las listas que
generó el proyecto y mide, por segmento, severidad y trayectoria, qué fracción de los clientes
marcados tenía de verdad algo que gestionar.

## Cómo alimentarlo

1. Copiar `gestion_caida\retroalimentacion\resultado_gestion_plantilla.csv` como
   `resultado_gestion_<lo que sea>.csv` en esa misma carpeta (se leen todos los `resultado_gestion*.csv`,
   la plantilla no).
2. Una fila por cliente visitado: `NIU`, `fecha_corte_lista` (AAAA-MM, el corte de la lista de donde
   salió), `fecha_visita`, `hallazgo` (del catálogo de abajo), `accion`, `observaciones`.
3. Correr este notebook (o dejar que lo corra `pipeline_mensual.py`, que lo incluye).

Mientras no haya archivos, el notebook lo dice y termina sin error.

In [ ]:
# ============================================================
# 1. LIBRERÍAS, RUTAS Y CATÁLOGO DE HALLAZGOS
# ============================================================

from pathlib import Path
import os
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

BASE_DIR = Path(os.environ.get("EBSA_DATOS", r"C:\Users\Home\Documents\Datos_Ebsa"))

GESTION_DIR = BASE_DIR / "07_gestion_caida"
HISTORIAL_GESTION_DIR = GESTION_DIR / "historial"
RETRO_DIR = GESTION_DIR / "retroalimentacion"
RETRO_DIR.mkdir(parents=True, exist_ok=True)

RUTA_OPERATIVA_ACTUAL = GESTION_DIR / "gestion_caida_operativa.csv"
RUTA_EVAL_RESUMEN = RETRO_DIR / "evaluacion_retroalimentacion_resumen.csv"
RUTA_EVAL_DETALLE = RETRO_DIR / "evaluacion_retroalimentacion_detalle.csv"

# --- Catálogo de hallazgos ---------------------------------------------------
# Tres grupos. Lo que importa para medir la lista:
#   CON_CAUSA_GESTIONABLE : había un problema que la empresa puede corregir y
#                           recuperar consumo/facturación. Es el acierto pleno.
#   CAIDA_REAL_SIN_GESTION: el cliente sí consume menos, pero no hay nada que
#                           reparar. La lista no se equivocó al marcarlo; solo
#                           no hay recuperación.
#   SIN_NOVEDAD           : consumo normal al visitar. Falso positivo.
#   NO_VERIFICADO         : no se pudo concluir. No cuenta ni a favor ni en contra.
# Se puede ampliar el catálogo; lo que no esté aquí se marca como OTRO y se avisa.
CATALOGO_HALLAZGOS = {
    "MEDIDOR_DANADO": "CON_CAUSA_GESTIONABLE",
    "MEDIDOR_MANIPULADO_O_FRAUDE": "CON_CAUSA_GESTIONABLE",
    "ERROR_DE_LECTURA_O_FACTURACION": "CON_CAUSA_GESTIONABLE",
    "ACOMETIDA_O_RED_CON_FALLA": "CON_CAUSA_GESTIONABLE",
    "CONEXION_IRREGULAR": "CON_CAUSA_GESTIONABLE",
    "PREDIO_DESOCUPADO": "CAIDA_REAL_SIN_GESTION",
    "CAMBIO_DE_ACTIVIDAD_O_HABITO": "CAIDA_REAL_SIN_GESTION",
    "AUTOGENERACION": "CAIDA_REAL_SIN_GESTION",
    "CLIENTE_RETIRADO": "CAIDA_REAL_SIN_GESTION",
    "SIN_NOVEDAD": "SIN_NOVEDAD",
    "NO_UBICADO": "NO_VERIFICADO",
    "PENDIENTE": "NO_VERIFICADO",
}

# Plantilla para que la empresa registre las visitas (se crea si no existe)
RUTA_PLANTILLA = RETRO_DIR / "resultado_gestion_plantilla.csv"
if not RUTA_PLANTILLA.exists():
    RUTA_PLANTILLA.write_text(
        "NIU,fecha_corte_lista,fecha_visita,hallazgo,accion,observaciones\n"
        "# Borra estas líneas de ejemplo (las que empiezan por #) y llena una fila por cliente visitado.\n"
        "# NIU: el del cliente tal como aparece en la lista. fecha_corte_lista: el corte de la lista (columna fecha_corte, formato AAAA-MM).\n"
        "# fecha_visita: AAAA-MM-DD. hallazgo: uno de los valores del catálogo de abajo. accion y observaciones: texto libre.\n"
        "#   " + " | ".join(CATALOGO_HALLAZGOS) + "\n",
        encoding="utf-8-sig",
    )
    print("Plantilla creada:", RUTA_PLANTILLA)

print("Retroalimentación:", RETRO_DIR)
print("Historial listas :", HISTORIAL_GESTION_DIR)
print("\nCatálogo de hallazgos:")
for k, v in CATALOGO_HALLAZGOS.items():
    print(f"  {k:<32} -> {v}")


In [ ]:
# ============================================================
# 2. LEER LOS ARCHIVOS DE RESULTADOS DE CAMPO
# ============================================================

archivos = sorted(
    r for r in RETRO_DIR.glob("resultado_gestion*.csv") if "plantilla" not in r.name.lower()
)

COLUMNAS_OBLIGATORIAS = ["NIU", "fecha_corte_lista", "hallazgo"]

partes = []
for ruta in archivos:
    df = pd.read_csv(ruta, dtype="string", comment="#", encoding="utf-8-sig", skip_blank_lines=True)
    df.columns = [c.strip() for c in df.columns]
    faltan = [c for c in COLUMNAS_OBLIGATORIAS if c not in df.columns]
    if faltan:
        raise ValueError(f"A {ruta.name} le faltan columnas obligatorias: {faltan}")
    df["archivo_origen"] = ruta.name
    partes.append(df)
    print(f"  {ruta.name}: {len(df):,} filas")

HAY_DATOS = bool(partes)

if not HAY_DATOS:
    print("No hay archivos resultado_gestion*.csv en", RETRO_DIR)
    print("Cuando la empresa registre visitas, se llena la plantilla y este notebook las evalúa.")
    retro = pd.DataFrame(columns=COLUMNAS_OBLIGATORIAS)
else:
    retro = pd.concat(partes, ignore_index=True)
    for c in ["NIU", "hallazgo", "fecha_corte_lista"]:
        retro[c] = retro[c].astype("string").str.strip()
    retro = retro[retro["NIU"].notna() & (retro["NIU"] != "")].copy()
    retro["hallazgo"] = retro["hallazgo"].str.upper().str.replace(" ", "_", regex=False)
    retro["fecha_corte_lista"] = retro["fecha_corte_lista"].str.slice(0, 7)

    desconocidos = sorted(set(retro["hallazgo"].dropna()) - set(CATALOGO_HALLAZGOS))
    if desconocidos:
        print(f"\n⚠ Hallazgos fuera del catálogo (se cuentan como OTRO / NO_VERIFICADO): {desconocidos}")
    retro["grupo_hallazgo"] = retro["hallazgo"].map(CATALOGO_HALLAZGOS).fillna("NO_VERIFICADO")

    if "fecha_visita" in retro.columns:
        retro["fecha_visita"] = pd.to_datetime(retro["fecha_visita"], errors="coerce")
        retro = retro.sort_values("fecha_visita")
    # Si un cliente fue visitado más de una vez para la misma lista, vale la última visita
    antes = len(retro)
    retro = retro.drop_duplicates(subset=["NIU", "fecha_corte_lista"], keep="last")
    if len(retro) < antes:
        print(f"  ({antes - len(retro):,} filas duplicadas NIU + corte: se conserva la última visita)")

    print(f"\nVisitas registradas: {len(retro):,} clientes en {retro['fecha_corte_lista'].nunique()} corte(s)")
    display(retro["grupo_hallazgo"].value_counts().rename("n").to_frame())


In [ ]:
# ============================================================
# 3. CRUZAR CON LA LISTA DE LA QUE SALIÓ CADA CLIENTE
# ============================================================

def cargar_lista(corte: str) -> pd.DataFrame:
    ruta = HISTORIAL_GESTION_DIR / f"gestion_caida_operativa_corte_{corte}.csv"
    if not ruta.exists():
        # Compatibilidad: si el corte pedido es el actual y aún no hay historial
        if RUTA_OPERATIVA_ACTUAL.exists():
            actual = pd.read_csv(RUTA_OPERATIVA_ACTUAL, dtype={"NIU": "string", "ciclo_etiqueta": "string"})
            if "fecha_corte" in actual.columns and str(actual["fecha_corte"].iloc[0])[:7] == corte:
                return actual
        raise FileNotFoundError(
            f"No existe la lista del corte {corte}:\n{ruta}\n"
            "Revisa fecha_corte_lista en el archivo de resultados."
        )
    return pd.read_csv(ruta, dtype={"NIU": "string", "ciclo_etiqueta": "string"})


if HAY_DATOS:
    cruces = []
    for corte, grupo in retro.groupby("fecha_corte_lista"):
        lista = cargar_lista(corte)
        lista["NIU"] = lista["NIU"].astype("string").str.strip()
        cols = [c for c in ["NIU", "ciclo_etiqueta", "zona", "clase_servicio", "estrato", "tramo_consumo",
                            "cluster_id", "veredicto", "severidad", "trayectoria", "estado_en_lista",
                            "perdida_kwh_mes", "valor_riesgo_mes", "orden_en_ciclo"] if c in lista.columns]
        cruce = grupo.merge(lista[cols], on="NIU", how="left", indicator=True)
        no_estaban = cruce["_merge"].eq("left_only")
        if no_estaban.any():
            print(f"  ⚠ corte {corte}: {int(no_estaban.sum()):,} NIU visitados no estaban en esa lista "
                  "(se evalúan aparte como 'FUERA_DE_LISTA')")
        cruce["en_lista"] = ~no_estaban
        cruce["cobertura_lista_pct"] = round(cruce["en_lista"].sum() / len(lista) * 100, 2)
        cruce["n_lista"] = len(lista)
        cruces.append(cruce.drop(columns="_merge"))
    detalle = pd.concat(cruces, ignore_index=True)

    # Acierto pleno y acierto amplio, solo sobre lo verificado
    detalle["verificado"] = detalle["grupo_hallazgo"].ne("NO_VERIFICADO")
    detalle["acierto_gestionable"] = detalle["grupo_hallazgo"].eq("CON_CAUSA_GESTIONABLE")
    detalle["caida_real"] = detalle["grupo_hallazgo"].isin(["CON_CAUSA_GESTIONABLE", "CAIDA_REAL_SIN_GESTION"])
    detalle["falso_positivo"] = detalle["grupo_hallazgo"].eq("SIN_NOVEDAD")

    detalle.to_csv(RUTA_EVAL_DETALLE, index=False, encoding="utf-8-sig")
    print("\nDetalle guardado:", RUTA_EVAL_DETALLE)
    display(detalle.head(10))


In [ ]:
# ============================================================
# 4. PRECISIÓN DE LA LISTA POR CORTE Y POR CADA FORMA DE MIRARLA
# ============================================================

def resumen_por(df, claves, etiqueta):
    base = df[df["en_lista"]]
    g = base.groupby(claves)
    out = pd.DataFrame({
        "visitados": g.size(),
        "verificados": g["verificado"].sum(),
        "con_causa_gestionable": g["acierto_gestionable"].sum(),
        "caida_real_sin_gestion": g["caida_real"].sum() - g["acierto_gestionable"].sum(),
        "sin_novedad": g["falso_positivo"].sum(),
        "valor_riesgo_gestionable_mes": g.apply(
            lambda x: x.loc[x["acierto_gestionable"], "valor_riesgo_mes"].sum()
            if "valor_riesgo_mes" in x.columns else np.nan, include_groups=False),
    }).reset_index()
    out["precision_gestionable_pct"] = (out["con_causa_gestionable"] / out["verificados"] * 100).round(1)
    out["precision_caida_real_pct"] = ((out["con_causa_gestionable"] + out["caida_real_sin_gestion"])
                                       / out["verificados"] * 100).round(1)
    out["corte_por"] = etiqueta
    out["valor_corte"] = out[claves[-1]].astype(str) if len(claves) > 1 else "TOTAL"
    return out[["corte_por", "valor_corte", "fecha_corte_lista", "visitados", "verificados",
                "con_causa_gestionable", "caida_real_sin_gestion", "sin_novedad",
                "precision_gestionable_pct", "precision_caida_real_pct", "valor_riesgo_gestionable_mes"]]


if HAY_DATOS and detalle["en_lista"].any():
    tablas = [resumen_por(detalle, ["fecha_corte_lista"], "TOTAL")]
    for col in ["severidad", "trayectoria", "estado_en_lista", "tramo_consumo", "zona",
                "cluster_id", "clase_servicio", "ciclo_etiqueta"]:
        if col in detalle.columns and detalle[col].notna().any():
            tablas.append(resumen_por(detalle, ["fecha_corte_lista", col], col))
    resumen = pd.concat(tablas, ignore_index=True)
    resumen.to_csv(RUTA_EVAL_RESUMEN, index=False, encoding="utf-8-sig")

    print("PRECISIÓN DE LA LISTA")
    print("=" * 78)
    print("  precision_gestionable_pct : % de verificados con un problema que la empresa puede corregir")
    print("  precision_caida_real_pct  : % de verificados donde la caída era real (gestionable o no)")
    print("  sin_novedad               : falsos positivos (consumo normal al visitar)")
    print()
    cobertura = detalle[detalle["en_lista"]].groupby("fecha_corte_lista").agg(
        n_lista=("n_lista", "first"), visitados=("NIU", "size"))
    cobertura["cobertura_pct"] = (cobertura["visitados"] / cobertura["n_lista"] * 100).round(2)
    print("Cobertura de la lista por corte:")
    display(cobertura)

    print("\nTOTAL POR CORTE")
    display(resumen[resumen["corte_por"] == "TOTAL"])
    for col in ["severidad", "trayectoria", "estado_en_lista", "tramo_consumo", "zona"]:
        t = resumen[resumen["corte_por"] == col]
        if len(t):
            print(f"\nPOR {col.upper()}")
            display(t.drop(columns=["corte_por"]))

    print("\nGuardado:", RUTA_EVAL_RESUMEN)

    # Gráfica: precisión por trayectoria y severidad (último corte)
    ultimo = resumen["fecha_corte_lista"].max()
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    for ax, col in zip(axes, ["trayectoria", "severidad"]):
        t = resumen[(resumen["corte_por"] == col) & (resumen["fecha_corte_lista"] == ultimo)]
        if len(t):
            ax.bar(t["valor_corte"], t["precision_gestionable_pct"], label="gestionable")
            ax.bar(t["valor_corte"], t["precision_caida_real_pct"] - t["precision_gestionable_pct"],
                   bottom=t["precision_gestionable_pct"], alpha=0.5, label="caída real sin gestión")
            ax.set_title(f"Precisión por {col} — corte {ultimo}")
            ax.set_ylabel("% de verificados")
            ax.tick_params(axis="x", rotation=30)
            ax.legend()
            ax.grid(alpha=0.25, axis="y")
    plt.tight_layout()
    plt.show()
elif HAY_DATOS:
    print("Ninguno de los NIU visitados estaba en las listas indicadas: revisa fecha_corte_lista.")
else:
    print("Sin datos de campo todavía: nada que evaluar.")


In [ ]:
# ============================================================
# 5. CIERRE
# ============================================================
print("EVALUACIÓN DE RETROALIMENTACIÓN — TERMINADA")
print("=" * 78)
if HAY_DATOS:
    print(f"Visitas evaluadas : {len(detalle):,} ({int(detalle['en_lista'].sum()):,} en lista)")
    print("Salidas:")
    print(" •", RUTA_EVAL_RESUMEN)
    print(" •", RUTA_EVAL_DETALLE)
    print("\nQué hacer con esto: si un cruce (p. ej. severidad MODERADA, o una trayectoria)")
    print("muestra precisión baja de forma sostenida, ese grupo debe bajar en el orden de la")
    print("lista o salir de ella. Ese ajuste se hace en Priorizacion_gestion_caida.ipynb.")
else:
    print("Sin archivos de resultados. Plantilla:", RETRO_DIR / "resultado_gestion_plantilla.csv")
